In [5]:
# ---------------------------------------
# Sudoku CSP Solver (CORRECTED VERSION)
# ---------------------------------------

backtrackCalls = 0
backtrackFailures = 0

# -------------------------------
# Read Board
# -------------------------------
def readBoard(fileName):
    board = []
    with open(fileName, "r") as file:
        for line in file:
            board.append([int(x) for x in line.strip()])
    return board

# -------------------------------
# Print Board
# -------------------------------
def printBoard(board):
    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("-" * 21)
        for j in range(9):
            if j % 3 == 0 and j != 0:
                print("|", end=" ")
            print(board[i][j], end=" ")
        print()
    print()

# -------------------------------
# Initialize Domains
# -------------------------------
def initializeDomains(board):
    domains = {}
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                domains[(row, col)] = set(range(1, 10))
            else:
                domains[(row, col)] = {board[row][col]}
    return domains

# -------------------------------
# Get Neighbors
# -------------------------------
def getNeighbors(row, col):
    neighbors = set()
    
    # Same row
    for i in range(9):
        if i != col:
            neighbors.add((row, i))
    
    # Same column
    for i in range(9):
        if i != row:
            neighbors.add((i, col))
    
    # Same box
    startRow = (row // 3) * 3
    startCol = (col // 3) * 3
    for i in range(3):
        for j in range(3):
            r = startRow + i
            c = startCol + j
            if (r, c) != (row, col):
                neighbors.add((r, c))
    
    return neighbors

# -------------------------------
# AC-3 Algorithm (CORRECTED)
# -------------------------------
def ac3(domains):
    from collections import deque
    queue = deque()
    
    # Initialize queue with all arcs
    for xi in domains:
        for xj in getNeighbors(xi[0], xi[1]):
            queue.append((xi, xj))
    
    while queue:
        xi, xj = queue.popleft()
        
        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            
            # Add all neighbors of xi except xj back to queue
            for xk in getNeighbors(xi[0], xi[1]):
                if xk != xj:
                    queue.append((xk, xi))
    
    return True

# -------------------------------
# Revise Function (CORRECTED)
# -------------------------------
def revise(domains, xi, xj):
    revised = False
    to_remove = []
    
    for value in domains[xi]:
        # Check if value has NO support in xj's domain
        has_support = False
        for other_value in domains[xj]:
            if value != other_value:
                has_support = True
                break
        
        if not has_support:
            to_remove.append(value)
    
    if to_remove:
        for value in to_remove:
            domains[xi].remove(value)
        revised = True
    
    return revised

# -------------------------------
# Check if assignment is consistent
# -------------------------------
def isConsistent(board, row, col, value):
    # Check row
    for c in range(9):
        if c != col and board[row][c] == value:
            return False
    
    # Check column
    for r in range(9):
        if r != row and board[r][col] == value:
            return False
    
    # Check box
    startRow = (row // 3) * 3
    startCol = (col // 3) * 3
    for r in range(startRow, startRow + 3):
        for c in range(startCol, startCol + 3):
            if (r, c) != (row, col) and board[r][c] == value:
                return False
    
    return True

# -------------------------------
# Select Unassigned Variable with MRV Heuristic
# -------------------------------
def selectUnassigned(domains, board):
    best_cell = None
    min_domain_size = 10
    
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                domain_size = len(domains.get((row, col), set()))
                if domain_size < min_domain_size:
                    min_domain_size = domain_size
                    best_cell = (row, col)
                    if min_domain_size == 1:
                        return best_cell
    
    return best_cell

# -------------------------------
# Backtracking with proper copying
# -------------------------------
def backtrack(board, domains):
    global backtrackCalls, backtrackFailures
    backtrackCalls += 1
    
    # Select unassigned variable
    cell = selectUnassigned(domains, board)
    if cell is None:
        return True
    
    row, col = cell
    
    # Try values in order
    for value in sorted(domains[(row, col)]):  # Try smaller values first
        
        if isConsistent(board, row, col, value):
            # Create deep copies for backtracking
            board_copy = [r[:] for r in board]
            domains_copy = {k: set(v) for k, v in domains.items()}
            
            # Make assignment
            board[row][col] = value
            domains[(row, col)] = {value}
            
            # Update domains of neighbors (forward checking)
            consistent = True
            for neighbor in getNeighbors(row, col):
                if board[neighbor[0]][neighbor[1]] == 0:
                    if value in domains[neighbor]:
                        domains[neighbor].remove(value)
                        if len(domains[neighbor]) == 0:
                            consistent = False
                            break
            
            if consistent:
                # Apply AC-3 for further constraint propagation
                if ac3(domains):
                    if backtrack(board, domains):
                        return True
            
            # Backtrack
            board = board_copy
            domains = domains_copy
    
    backtrackFailures += 1
    return False

# -------------------------------
# Solve Function
# -------------------------------
def solveSudoku(fileName):
    global backtrackCalls, backtrackFailures
    
    backtrackCalls = 0
    backtrackFailures = 0
    
    board = readBoard(fileName)
    domains = initializeDomains(board)
    
    print(f"\n{'='*50}")
    print(f"Solving: {fileName}")
    print('='*50)
    print("Original Board:")
    printBoard(board)
    
    # Initial AC-3 to prune domains
    if not ac3(domains):
        print("No solution found (inconsistent domains)")
        return
    
    if backtrack(board, domains):
        print("✓ Solved Board:")
        printBoard(board)
        print(f"Statistics:")
        print(f"  Backtrack Calls: {backtrackCalls}")
        print(f"  Backtrack Failures: {backtrackFailures}")
        if backtrackCalls > 0:
            success_rate = (backtrackCalls - backtrackFailures) / backtrackCalls * 100
            print(f"  Success Rate: {success_rate:.2f}%")
    else:
        print("✗ No solution found")
        print(f"Backtrack Calls attempted: {backtrackCalls}")

# -------------------------------
# Create sample puzzle files if they don't exist
# -------------------------------
# -------------------------------
# Run All Boards
# -------------------------------
if __name__ == "__main__":
    createSampleFiles()
    solveSudoku("easy.txt")
    solveSudoku("medium.txt")
    solveSudoku("hard.txt")
    solveSudoku("veryhard.txt")


Solving: easy.txt
Original Board:
0 0 4 | 0 3 0 | 0 5 0 
6 0 9 | 4 0 0 | 0 0 0 
0 0 5 | 1 0 0 | 4 8 9 
---------------------
0 0 0 | 0 6 0 | 9 3 0 
3 0 0 | 8 0 7 | 0 0 2 
0 2 6 | 0 4 0 | 0 0 0 
---------------------
4 5 3 | 0 0 9 | 6 0 0 
0 0 0 | 0 0 4 | 7 0 5 
0 9 0 | 0 5 0 | 2 0 0 

✓ Solved Board:
7 8 4 | 9 3 2 | 1 5 6 
6 1 9 | 4 8 5 | 3 2 7 
2 3 5 | 1 7 6 | 4 8 9 
---------------------
5 7 8 | 2 6 1 | 9 3 4 
3 4 1 | 8 9 7 | 5 6 2 
9 2 6 | 5 4 3 | 8 7 1 
---------------------
4 5 3 | 7 2 9 | 6 1 8 
8 6 2 | 3 1 4 | 7 9 5 
1 9 7 | 6 5 8 | 2 4 3 

Statistics:
  Backtrack Calls: 50
  Backtrack Failures: 0
  Success Rate: 100.00%

Solving: medium.txt
Original Board:
5 3 0 | 0 7 0 | 0 0 0 
6 0 0 | 1 9 5 | 0 0 0 
0 9 8 | 0 0 0 | 0 6 0 
---------------------
8 0 0 | 0 6 0 | 0 0 3 
4 0 0 | 8 0 3 | 0 0 1 
7 0 0 | 0 2 0 | 0 0 6 
---------------------
0 6 0 | 0 0 0 | 2 8 0 
0 0 0 | 4 1 9 | 0 0 5 
0 0 0 | 0 8 0 | 0 7 9 

✓ Solved Board:
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 